In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

generator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.8 
)


optimizer_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.7 
)

In [7]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final eveluation result.")
    feedback: str = Field(..., description="Constructive feedback for the tweet")



structured_evaluator_llm = generator_llm.with_structured_output(TweetEvaluation) 

In [5]:
# state
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "need_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [16]:
def generate_tweet(state: TweetState):
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer"),
        HumanMessage(content=f"""
            Write a short, original and hilarious tweet on the topic: "{state['topic']}".
            Rules:
            - Do not use question answer format.
            - Max 280 characters.
            - Use observational humor, irony, sarcasm or cultural refrences.
            - Think in meme logic, punchlines, or relateable takes.
            - use simple, day to day english.
        """)
    ]

    res = generator_llm.invoke(messages).content

    return {"tweet": res}

def evaluate_tweet(state: TweetState):
        messages = [
    SystemMessage(
        content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
    ),
    HumanMessage(
        content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-unclie universe" or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
"""
    ),
        ]


        res = structured_evaluator_llm.invoke(messages)

        return {"evaluation": res.evaluation, "feedback": res.feedback}


def optimize_tweet(state: TweetState):
    messages = [
    SystemMessage(
        content="You punch up tweets for virality and humor based on given feedback."
    ),
    HumanMessage(
        content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
"""
    ),
]


    res = optimizer_llm.invoke(messages).content

    iteration = state['iteration'] + 1
    return {"tweet": res, "iteration": iteration}

In [9]:
def route_evalution(state: TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return "approved"
    else:
        return "needs_improvement"

In [17]:
graph = StateGraph(TweetState)

graph.add_node('generate_tweet', generate_tweet)
graph.add_node('evaluate_tweet', evaluate_tweet)
graph.add_node('optimize_tweet', optimize_tweet)

graph.add_edge(START, 'generate_tweet')
graph.add_edge('generate_tweet', 'evaluate_tweet')

graph.add_conditional_edges('evaluate_tweet', route_evalution, {'approved': END, 'needs_improvement': 'optimize_tweet'})
graph.add_edge('optimize_tweet', 'evaluate_tweet')

workflow = graph.compile()

In [18]:
initial_state = {
    "topic": "Monkey D. Luffy",
    "iteration": 1,
    "max_iteration": 5
}

final_state = workflow.invoke(initial_state)

In [19]:
final_state

{'topic': 'Monkey D. Luffy',
 'tweet': 'Luffy wakes up, chooses violence, and then eats enough food to feed a small village. My entire grocery budget just ran for cover.',
 'evaluation': 'approved',
 'feedback': "This tweet expertly blends a specific pop culture reference with universally relatable financial humor, making it genuinely funny and highly shareable. The punchline about the grocery budget running for cover is original and lands perfectly, ensuring it's scroll-stopping without relying on tired joke formats. It's concise, well-formatted, and has strong virality potential, particularly among anime fans and anyone dealing with rising food costs.",
 'iteration': 1,
 'max_iteration': 5}